In [ ]:
import pandas as pd
import polars as pl
import os
import torch
import pairadigm
import dotenv

dotenv.load_dotenv(dotenv_path='/fs/clip-projects/clip-k12/mlchrzan/.env')

In [ ]:
# Load Data and Pairadigms
emobank = pd.read_csv('Data/emobank.csv')
emopair = pd.read_csv('Results/emopair.csv')
emopair_pairs = pd.read_csv("Results/emopair_pairs.csv")
validation_valence = pairadigm.load_pairadigm("Results/pairadigms/untied_valence_pairadigm_12-19-25.pkl")
validation_arousal = pairadigm.load_pairadigm("Results/pairadigms/untied_arousal_pairadigm_12-19-25.pkl")
validation_dominance = pairadigm.load_pairadigm("Results/pairadigms/untied_dominance_pairadigm_12-19-25.pkl")


# Annotator Validation

In [ ]:
altest_results = []
for construct in ['valence', 'arousal', 'dominance']:
    if construct == 'valence':
        _ , data = validation_valence.plot_epsilon_sensitivity(return_data=True);
    elif construct == 'dominance':
        _ , data = validation_dominance.plot_epsilon_sensitivity(return_data=True);
    elif construct == 'arousal':
        _ , data = validation_arousal.plot_epsilon_sensitivity(return_data=True);
    data['construct'] = construct
    altest_results.append(data)
    
altest_results = pd.concat(altest_results, ignore_index=True)
altest_results.sort_values(by=['construct', 'model', 'epsilon'], inplace=True)
altest_results

In [ ]:
altest_results

In [ ]:
# Since advantage probability doesn't change with episilon, we can just take the first value for each model and construct
ad_probs = altest_results.groupby(['construct', 'model']).first().reset_index()[['construct', 'model', 'advantage_prob']].copy()
ad_probs

In [ ]:
# ...existing code...
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="talk")

# Keep model order consistent across all facets
model_order = sorted(ad_probs["model"].unique())

# Make the constructs capitalized for better presentation
ad_probs["construct"] = ad_probs["construct"].str.capitalize()

# Force facet order: Valence, Arousal, Dominance
construct_order = ["Valence", "Arousal", "Dominance"]
ad_probs["construct"] = pd.Categorical(
    ad_probs["construct"],
    categories=construct_order,
    ordered=True
)

g = sns.catplot(
    data=ad_probs,
    y="model",                    # model names on y-axis
    x="advantage_prob",
    col="construct",
    col_order=construct_order,    # facet order
    kind="bar",
    order=model_order,            # shared categorical y-axis ordering
    height=4,
    aspect=1.2,
    palette="viridis",
    hue='model',
    legend=False,
)

g.set_axis_labels("", "Model")   # remove per-facet x-axis title
g.set_titles("{col_name}")

for ax in g.axes.flat:
    ax.set_xlim(0, 1.05)

    # Annotate each bar with probability rounded to hundredths
    for p in ax.patches:
        w = p.get_width()
        y = p.get_y() + p.get_height() / 2
        label = f"{w:.2f}"

        if w >= 0.95:
            ax.text(w - 0.03, y, label, va="center", ha="right", fontsize=10)
        else:
            ax.text(w + 0.01, y, label, va="center", ha="left", fontsize=10)

g.fig.suptitle("Advantage Probability by Model and Construct", y=1)
# g.fig.supxlabel("Advantage Probability")
plt.tight_layout()
plt.show()

In [ ]:
# Plot win rate over epsilon, faceted vertically by construct
from matplotlib.lines import Line2D

winning_rate_data = altest_results[['epsilon', 'winning_rate', 'model', 'construct']].drop_duplicates()
winning_rate_data["construct"] = winning_rate_data["construct"].str.capitalize()
winning_rate_data["construct"] = pd.Categorical(
    winning_rate_data["construct"],
    categories=construct_order,
    ordered=True
)

g2 = sns.relplot(
    data=winning_rate_data,
    x="epsilon",
    y="winning_rate",
    row="construct",
    row_order=construct_order,
    hue="model",
    hue_order=model_order,
    kind="line",
    height=4.2,          # taller facets (was 3.0)
    aspect=1.6,          # less wide (was 2.2)
    palette="viridis",
    legend=False,
)

g2.set_titles("{row_name}")
g2.set_axis_labels("", "Win Rate")

for ax in g2.axes.flat:
    ax.axhline(y=0.5, color="red", linestyle="--", linewidth=2, alpha=0.7)
    ax.set_ylim(-0.05, 1.05)
    ax.set_xlabel("")

palette = sns.color_palette("viridis", n_colors=len(model_order))
model_handles = [
    Line2D([0], [0], color=palette[i], lw=2, label=model)
    for i, model in enumerate(model_order)
]
threshold_handle = Line2D([0], [0], color="red", lw=2, linestyle="--", label="0.50 Threshold")

g2.fig.suptitle("Win Rate Sensitivity to Epsilon by Construct", y=0.995)
g2.fig.legend(
    handles=model_handles + [threshold_handle],
    title="model",
    loc="upper center",
    bbox_to_anchor=(0.5, 0.96),
    ncol=2,              # narrower legend (was 3)
    frameon=True,
    columnspacing=1.0,
    handlelength=2.0,
)

g2.fig.supxlabel("Epsilon", y=0.05)
plt.tight_layout(rect=[0, 0.04, 1, 0.88])  # more top room for title + legend
plt.show()

## IRRs

In [ ]:
irrs = []
for construct in ['valence', 'arousal', 'dominance']:
    if construct == 'valence':
        irr = validation_valence.irr()
        irr['construct'] = 'valence'
    elif construct == 'dominance':
        irr = validation_dominance.irr()
        irr['construct'] = 'dominance'
    elif construct == 'arousal':
        irr = validation_arousal.irr()
        irr['construct'] = 'arousal'
    irrs.append(irr)

irrs = pd.concat(irrs, ignore_index=True)
irrs

In [ ]:
irrs_human = irrs[irrs['group'] == 'human'].copy()

cols_to_select = ['construct', 'n_items', 'score']
irrs_human = irrs_human[cols_to_select].copy()

irrs_human.rename(columns={'score': 'emopair_irrs'}, inplace=True)
irrs_human['emobank_irrs'] = [0.738, 0.595, 0.570]

# Rearrange columns for better presentation
irrs_human = irrs_human[['construct', 'n_items', 'emobank_irrs', 'emopair_irrs']]

# Round emopair_irrs for better readability to the 1000ths place
irrs_human['emopair_irrs'] = irrs_human['emopair_irrs'].round(3)

# Rename Columns for Presentation
irrs_human.rename(columns={
    'construct': 'Construct',
    'n_items': 'Number of Items',
    'emobank_irrs': 'EmoBank IRR',
    'emopair_irrs': 'EmoPair IRR'
}, inplace=True)

# Capitalize Construct Names
irrs_human['Construct'] = irrs_human['Construct'].str.capitalize()

irrs_human['Improvement'] = (irrs_human['EmoPair IRR'] - irrs_human['EmoBank IRR']).round(3)

irrs_human.to_csv('Results/irrs_comparison.csv', index=False)
irrs_human

# Score Distributions

In [ ]:
emopair

In [ ]:
emopair

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")

df = emopair.copy()

# Calculate Spearman rank correlations
aro_corr, aro_pval = spearmanr(df['A_rescaled'], df['A_BT_score_full'])
dom_corr, dom_pval = spearmanr(df['D_rescaled'], df['D_BT_score_full'])

# Create figure with subplots
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
sns.set_style("whitegrid")

# Arousal plot
ax1 = axes[0]
ax1.hist(df['A_rescaled'], bins=25, alpha=0.6, label='Rescaled EmoBank', color='#2E86AB', edgecolor='black', linewidth=0.5)
ax1.hist(df['A_BT_score_full'], bins=25, alpha=0.6, label='EmoPair', color='#F18F01', edgecolor='black', linewidth=0.5)
ax1.set_xlabel('Score', fontsize=12)
ax1.set_ylabel('Frequency', fontsize=12)
ax1.set_title('Arousal', fontsize=13, pad=15)
ax1.grid(True, alpha=0.2, linestyle='--', axis='y')
ax1.legend(loc='upper right', framealpha=0.9, fontsize=10, frameon=True)

# Add correlation text box
textstr_aro = f"Spearman ρ = {aro_corr:.3f}\np < 0.001***" if aro_pval < 0.001 else f"Spearman ρ = {aro_corr:.3f}\np = {aro_pval:.4f}"
ax1.text(0.05, 0.95, textstr_aro, transform=ax1.transAxes, fontsize=11,
         verticalalignment='top', bbox=dict(boxstyle='round', facecolor='#F0F0F0', alpha=0.9, pad=0.8, edgecolor='gray', linewidth=0.5))

# Dominance plot
ax2 = axes[1]
ax2.hist(df['D_rescaled'], bins=25, alpha=0.6, label='Rescaled EmoBank', color='#A23B72', edgecolor='black', linewidth=0.5)
ax2.hist(df['D_BT_score_full'], bins=25, alpha=0.6, label='EmoPair', color='#F18F01', edgecolor='black', linewidth=0.5)
ax2.set_xlabel('Score', fontsize=12)
ax2.set_ylabel('Frequency', fontsize=12)
ax2.set_title('Dominance', fontsize=13, pad=15)
ax2.grid(True, alpha=0.2, linestyle='--', axis='y')
ax2.legend(loc='upper right', framealpha=0.9, fontsize=10, frameon=True)

# Add correlation text box
textstr_dom = f"Spearman ρ = {dom_corr:.3f}\np < 0.001***" if dom_pval < 0.001 else f"Spearman ρ = {dom_corr:.3f}\np = {dom_pval:.4f}"
ax2.text(0.05, 0.95, textstr_dom, transform=ax2.transAxes, fontsize=11,
         verticalalignment='top', bbox=dict(boxstyle='round', facecolor='#F0F0F0', alpha=0.9, pad=0.8, edgecolor='gray', linewidth=0.5))

fig.suptitle('Comparison of EmoPair Scores Against Original EmoBank Scores', fontsize=14, y=0.98)

plt.tight_layout()
plt.savefig('Results/visuals/bt_score_validationv2.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="talk")

df = emopair.copy()

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

dim_configs = {
    "Arousal": {
        "full": "A_BT_score_full",
        "split": "A_BT_score_split",
        "color_full": "#2E86AB",
        "color_split": "#F18F01",
        "ax": axes[0],
    },
    "Dominance": {
        "full": "D_BT_score_full",
        "split": "D_BT_score_split",
        "color_full": "#A23B72",
        "color_split": "#F18F01",
        "ax": axes[1],
    },
}

for dim, cfg in dim_configs.items():
    ax = cfg["ax"]
    ax.hist(df[cfg["full"]], bins=25, alpha=0.6, label="Full", color=cfg["color_full"], edgecolor="black", linewidth=0.5)
    ax.hist(df[cfg["split"]], bins=25, alpha=0.6, label="Split", color=cfg["color_split"], edgecolor="black", linewidth=0.5)
    ax.set_title(dim, fontsize=13, pad=12)
    ax.set_xlabel("BT Score", fontsize=12)
    ax.set_ylabel("Frequency", fontsize=12)
    ax.legend(loc="upper right", framealpha=0.9, fontsize=10)
    ax.grid(True, alpha=0.2, linestyle="--", axis="y")

fig.suptitle("Full vs. Split BT Score Distributions by Dimension", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig("Results/visuals/bt_score_full_vs_split.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

sns.set_theme(style="whitegrid", context="talk")

df = emopair.copy()

split_order = ["train", "dev", "test"]
dim_configs = {
    "Arousal": {
        "full": "A_BT_score_full",
        "split": "A_BT_score_split",
        "color_full": "#2E86AB",
        "color_split": "#F18F01",
        "row": 0,
    },
    "Dominance": {
        "full": "D_BT_score_full",
        "split": "D_BT_score_split",
        "color_full": "#A23B72",
        "color_split": "#F18F01",
        "row": 1,
    },
}

# Rows: Arousal, Dominance | Cols: train, dev, test
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for dim, cfg in dim_configs.items():
    for col, split_name in enumerate(split_order):
        ax = axes[cfg["row"]][col]
        subset = df[df["split"] == split_name]

        ax.hist(subset[cfg["full"]], bins=20, alpha=0.6, label="Full",
                color=cfg["color_full"], edgecolor="black", linewidth=0.5)
        ax.hist(subset[cfg["split"]], bins=20, alpha=0.6, label="Split",
                color=cfg["color_split"], edgecolor="black", linewidth=0.5)

        ax.set_title(f"{dim} — {split_name.capitalize()}", fontsize=13, pad=10)
        ax.set_xlabel("BT Score", fontsize=11)
        ax.set_ylabel("Frequency", fontsize=11)
        ax.grid(True, alpha=0.2, linestyle="--", axis="y")
        ax.legend(loc="upper right", framealpha=0.9, fontsize=10)

        # Spearman correlation annotation
        corr, pval = spearmanr(subset[cfg["full"]], subset[cfg["split"]])
        pval_str = "p < 0.001***" if pval < 0.001 else f"p = {pval:.4f}"
        textstr = f"Spearman ρ = {corr:.3f}\n{pval_str}"
        ax.text(0.05, 0.95, textstr, transform=ax.transAxes, fontsize=9,
                verticalalignment="top",
                bbox=dict(boxstyle="round", facecolor="#F0F0F0", alpha=0.9,
                          edgecolor="gray", linewidth=0.5))

fig.suptitle("Full vs. Split BT Score Distributions by Dimension and Data Split",
             fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig("Results/visuals/bt_score_full_vs_split_by_datasplit.png",
            dpi=300, bbox_inches="tight", facecolor="white")
plt.show()